# spark.read — read a CSV (try it live)

Runnable companion to the note: **[DataFrameReader](https://ravi-writes.pages.dev/notes/abinitio-to-pyspark/core-operations/dataframereader)**.

`spark.read` is Ab Initio's **Input File / Input Table**. We make a file with plain Python + Faker, then read it. Run top to bottom.

## The data & the requirement

No file yet, so first we make one with **plain Python + Faker** — a true external input (using Spark to make the file we then read with Spark would be circular). Inputs live under `/content/ravi-writes/data/input/` (`/content` is Colab's ephemeral disk).

**Requirement:** load that CSV into a Spark DataFrame with the **right column types** — not everything as text.

In [ ]:
!pip install -q faker

In [ ]:
import csv, os, random
from faker import Faker

INPUT_PATH = "/content/ravi-writes/data/input/customers.csv"
os.makedirs(os.path.dirname(INPUT_PATH), exist_ok=True)

fake = Faker(); Faker.seed(42); random.seed(42)   # reproducible — same rows every run
with open(INPUT_PATH, "w", newline="") as f:
    writer = csv.writer(f)
    writer.writerow(["id", "name", "signup_date", "balance"])
    for i in range(1, 9):        # 8 customers
        writer.writerow([
            i,
            fake.name(),
            fake.date_between(start_date="-2y", end_date="today").isoformat(),
            round(random.uniform(0, 500), 2),
        ])

print(open(INPUT_PATH).read())   # it's just plain text on disk

## The Ab Initio solution

Reading a file is the **Input File** component — it pulls the external `customers.csv` into the graph:

```text
  customers.csv  ──(spark.read)──►  Input File  ──►  DataFrame
  (external, from plain Python)
```

## The Spark solution — step by step

In [ ]:
# 1. Install PySpark (one-time, in Colab's runtime)
!pip install -q pyspark

In [ ]:
# 2. Imports & SparkSession
from pyspark.sql import SparkSession

spark = SparkSession.builder.appName("read-demo").getOrCreate()

In [ ]:
# 3. Read with no schema -> every column comes in as a string
raw = spark.read.option("header", True).csv(INPUT_PATH)
raw.printSchema()

In [ ]:
# 4. Read with an explicit schema (the Ab Initio DML) -> real types
from pyspark.sql.types import (
    StructType, StructField, IntegerType, StringType, DoubleType,
)

schema = StructType([
    StructField("id",          IntegerType(), nullable=True),
    StructField("name",        StringType(),  nullable=True),
    StructField("signup_date", StringType(),  nullable=True),
    StructField("balance",     DoubleType(),  nullable=True),
])

df = spark.read.schema(schema).option("header", True).csv(INPUT_PATH)
df.show()
df.printSchema()

## Your turn

Try these yourself:

1. **Let Spark guess** — read with `.option("inferSchema", True)` (no `.schema(...)`), then `printSchema()`. What did it get right?
2. **Generic form** — rewrite as `spark.read.format("csv").schema(schema).option("header", True).load(INPUT_PATH)` and confirm it matches.
3. **Scale up** — bump the loop to `range(1, 1001)` for 1,000 customers, re-run, and try `df.count()`.